#  Toyota Stock Price Prediction — XGBoost (Lag Features)

**Dataset:** `Toyota_Stock_Prices_1980_2026.csv`  
**Task:** Regression dengan Lag Features  
**Model:** XGBoost Regressor

---
###  Daftar Isi
1. Cara Melihat Tipe Data
2. Dataset Bisa Digunakan Untuk Apa
3. Model Yang Bisa Digunakan
4. Parameter Yang Bisa Diubah/Disetel
5. Evaluasi Yang Dipakai
6. Cara Mengetahui Evaluasi Bagus atau Tidak
7. Cara Mengoptimasi Model
8. Cara Menyimpan Model
9. Cara Menggunakan Model Hasil Training

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib, os, warnings
warnings.filterwarnings('ignore')

from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

print('Libraries loaded ')
print(f'XGBoost: {xgb.__version__}')

---
## 1.  Cara Melihat Tipe Data

In [ ]:
df = pd.read_csv('../Toyota_Stock_Prices_1980_2026.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
print(f'Shape: {df.shape}')
df.head()

---
## 2.  Dataset Bisa Digunakan Untuk Apa

**Strategi:** Konversi time series ke supervised learning dengan **lag features + technical indicators**.

### Technical Indicators yang digunakan:
| Indikator | Keterangan |
|-----------|------------|
| MA (Moving Average) | Smoothing tren |
| EMA (Exponential MA) | Lebih sensitif terhadap perubahan baru |
| RSI | Relative Strength Index — momentum overbought/oversold |
| Bollinger Bands | Volatilitas |
| Rate of Change | Persentase perubahan dalam N hari |

In [ ]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = (-delta.clip(upper=0)).rolling(period).mean()
    rs = gain / (loss + 1e-9)
    return 100 - (100 / (1 + rs))

def create_features(df_in, lags=[1,2,3,5,10,20,30]):
    d = df_in.copy()
    
    for lag in lags:
        d[f'close_lag_{lag}'] = d['Close'].shift(lag)
        d[f'volume_lag_{lag}'] = d['Volume'].shift(lag)
    
    # Moving averages
    for w in [5, 10, 20, 50]:
        d[f'ma_{w}'] = d['Close'].rolling(w).mean()
        d[f'ema_{w}'] = d['Close'].ewm(span=w).mean()
    
    # RSI
    d['rsi_14'] = compute_rsi(d['Close'])
    
    # Volatility
    d['vol_5'] = d['Close'].rolling(5).std()
    d['vol_20'] = d['Close'].rolling(20).std()
    
    # Rate of change
    for n in [1, 5, 20]:
        d[f'roc_{n}'] = d['Close'].pct_change(n)
    
    # Price features
    d['daily_range'] = d['High'] - d['Low']
    d['price_position'] = (d['Close'] - d['Low']) / (d['High'] - d['Low'] + 1e-8)
    
    # Time features
    d['day_of_week'] = d['Date'].dt.dayofweek
    d['month'] = d['Date'].dt.month
    d['quarter'] = d['Date'].dt.quarter
    d['year'] = d['Date'].dt.year
    
    # Target: harga besok
    d['target'] = d['Close'].shift(-1)
    
    return d.dropna()

df_feat = create_features(df[df['Date'] >= '2010-01-01'].copy())
print(f'Dataset setelah feature engineering: {df_feat.shape}')

In [ ]:
exclude = ['Date', 'target', 'Close', 'Open', 'High', 'Low', 'Volume']
feature_cols = [c for c in df_feat.columns if c not in exclude]
X = df_feat[feature_cols].values
y = df_feat['target'].values

split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

---
## 3.  Kenapa XGBoost untuk Stock?

**XGBoost unggul dibanding Random Forest untuk data finansial:**
- Gradient boosting memperbaiki kesalahan prediksi secara iteratif
- Regularisasi L1/L2 bawaan mencegah overfitting
- Early stopping pada eval set
- Penanganan fitur yang hilang secara otomatis
- Umumnya memberikan RMSE lebih rendah dari Random Forest

In [ ]:
model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='reg:squarederror',
    eval_metric='rmse',
    early_stopping_rounds=30,
    random_state=42, n_jobs=-1, verbosity=0
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
print(f'Training selesai  (best iter: {model.best_iteration})')

---
## 4.  Parameter Yang Bisa Diubah / Disetel

| Parameter | Nilai | Penjelasan |
|-----------|-------|------------|
| `n_estimators` | 100–5000 | Banyak pohon (gunakan early stopping) |
| `learning_rate` | 0.001–0.3 | Lebih kecil lebih bagus tapi lebih lambat |
| `max_depth` | 3–8 | Kedalaman pohon |
| `subsample` | 0.6–1.0 | Stochastic subsampling |
| `colsample_bytree` | 0.6–1.0 | Feature subsampling |
| `min_child_weight` | 1–10 | Regularisasi leaf |
| `gamma` | 0–5 | Min gain untuk split |
| `reg_alpha` | 0–1 | L1 regularization |
| `reg_lambda` | 0–10 | L2 regularization |
| `early_stopping_rounds` | 10–50 | Berhenti jika val tidak membaik |

---
## 5.  Evaluasi Yang Dipakai

In [ ]:
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-8))) * 100
dir_acc = np.mean(np.sign(np.diff(y_test)) == np.sign(np.diff(y_pred))) * 100

print('='*55)
print(' EVALUASI XGBOOST — STOCK PREDICTION')
print('='*55)
print(f'MAE            : {mae:.4f}')
print(f'RMSE           : {rmse:.4f}')
print(f'R²             : {r2:.4f}')
print(f'MAPE           : {mape:.2f}%')
print(f'Directional Acc: {dir_acc:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(y_test[:200], label='Aktual', color='steelblue')
axes[0].plot(y_pred[:200], label='XGB Pred', color='orange', alpha=0.8)
axes[0].set_title('Aktual vs XGBoost (200 hari pertama test)')
axes[0].legend()

feat_imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(10)
feat_imp.plot(kind='barh', ax=axes[1], color='orange')
axes[1].set_title('Top 10 Feature Importance')
axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

---
## 6.  Cara Mengetahui Evaluasi Bagus atau Tidak

### Bandingkan dengan Naive Model:

In [ ]:
lag1_idx = list(feature_cols).index('close_lag_1') if 'close_lag_1' in feature_cols else 0
naive_pred = X_test[:, lag1_idx]
naive_rmse = np.sqrt(mean_squared_error(y_test, naive_pred))

print(f'Naive RMSE : {naive_rmse:.4f}')
print(f'XGBoost RMSE: {rmse:.4f}')
improvement = (naive_rmse - rmse) / naive_rmse * 100
if improvement > 5:
    print(f' XGBoost lebih baik {improvement:.1f}% dari naive baseline!')
elif improvement > 0:
    print(f' XGBoost sedikit lebih baik {improvement:.1f}%')
else:
    print(f' XGBoost lebih buruk dari naive baseline — evaluasi ulang')

---
## 7.  Cara Mengoptimasi Model

### Tips Khusus untuk Stock dengan XGBoost:

1. **Feature engineering lebih dalam** — tambahkan MACD, Stochastic Oscillator, ATR
2. **Permintaan-pasokan sentiment** — tambahkan data news/earnings sebagai fitur
3. **Multi-step forecasting** — prediksi beberapa hari sekaligus dengan SHAP
4. **Walk-forward validation** — lebih realistis dari simple train/test split

In [ ]:
# Walk-forward validation
n_splits = 5
fold_size = len(X) // (n_splits + 1)
wf_scores = []

for i in range(1, n_splits + 1):
    tr_end = fold_size * i
    te_end = min(fold_size * (i + 1), len(X))
    X_tr, y_tr = X[:tr_end], y[:tr_end]
    X_te, y_te = X[tr_end:te_end], y[tr_end:te_end]
    
    m = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=5,
                     objective='reg:squarederror', verbosity=0, random_state=42)
    m.fit(X_tr, y_tr)
    yp = m.predict(X_te)
    fold_rmse = np.sqrt(mean_squared_error(y_te, yp))
    wf_scores.append(fold_rmse)
    print(f'Fold {i}: RMSE={fold_rmse:.4f}')

print(f'\nWalk-Forward mean RMSE: {np.mean(wf_scores):.4f} ± {np.std(wf_scores):.4f}')

---
## 8.  Cara Menyimpan Model

In [ ]:
os.makedirs('saved_models', exist_ok=True)
joblib.dump(model, 'saved_models/xgb_stock.pkl')
model.save_model('saved_models/xgb_stock.json')
joblib.dump(feature_cols, 'saved_models/feature_cols_xgb_stock.pkl')
print(' XGBoost stock model tersimpan!')

---
## 9.  Cara Menggunakan Model Hasil Training

In [ ]:
loaded_model = joblib.load('saved_models/xgb_stock.pkl')
loaded_cols = joblib.load('saved_models/feature_cols_xgb_stock.pkl')
print('Model dimuat ')

last_row = df_feat.tail(1)[loaded_cols]
pred_price = loaded_model.predict(last_row.values)[0]
last_price = df_feat['Close'].iloc[-1]

print(f'Harga Close terakhir : ${last_price:.2f}')
print(f'Prediksi harga besok : ${pred_price:.2f}')
chg = (pred_price - last_price) / last_price * 100
print(f'Perkiraan perubahan  : {chg:+.2f}%')
print(f'Sinyal               : {" BUY" if chg > 0.5 else (" SELL" if chg < -0.5 else "⏸ HOLD")}')